# 🟩 Cell 1 — Install & Imports

In [1]:

import cv2
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 🟩 Cell 2 — Initialize MediaPipe

In [2]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

detector = vision.HandLandmarker.create_from_options(options)

# 🟩 Cell 3 — Servo State + System State

In [3]:
# Servo angles (0–180)
servo = {
    "S1": 90,  # gripper
    "S2": 90,
    "S3": 90,
    "S4": 90   # base
}

# System states
lock_mode = False
active_servo = None
cooldown = 0

# Previous values (for motion tracking)
prev_wrist_x = None
prev_depth = None
prev_thumb_angle = None
prev_pinch_dist = None

# 🟩 Cell 4 — Gesture Detection Functions

In [4]:
# 🔒 Keep this for lock (unchanged)
def is_four_fingers(lm):
    tips = [8, 12, 16, 20]
    return sum(lm[tip].y < lm[tip - 2].y for tip in tips) == 4


# 🎯 Gesture Scoring Functions (0 → 1)

def score_pinch(lm):
    dist = get_distance(lm[4], lm[8])

    # also ensure other fingers NOT closed (avoid thumb confusion)
    other_open = (
        lm[12].y < lm[10].y or
        lm[16].y < lm[14].y or
        lm[20].y < lm[18].y
    )

    return max(0, 1 - dist * 12) * (1 if other_open else 0.5)


def score_three_fingers(lm):
    # Index, middle, ring OPEN
    index_open = lm[8].y < lm[6].y
    middle_open = lm[12].y < lm[10].y
    ring_open = lm[16].y < lm[14].y

    # Pinky CLOSED (important to distinguish from open hand)
    pinky_closed = lm[20].y > lm[18].y

    score = sum([
        index_open,
        middle_open,
        ring_open,
        pinky_closed
    ]) / 4

    return score


def score_open(lm):
    index_open  = lm[8].y < lm[6].y
    middle_open = lm[12].y < lm[10].y
    ring_open   = lm[16].y < lm[14].y
    pinky_open  = lm[20].y < lm[18].y

    return sum([index_open, middle_open, ring_open, pinky_open]) / 4


def score_thumb(lm):
    # Thumb extended
    thumb_extended = lm[4].x > lm[3].x

    # All other fingers MUST be closed
    index_closed  = lm[8].y > lm[6].y
    middle_closed = lm[12].y > lm[10].y
    ring_closed   = lm[16].y > lm[14].y
    pinky_closed  = lm[20].y > lm[18].y

    all_closed = index_closed and middle_closed and ring_closed and pinky_closed

    if thumb_extended and all_closed:
        return 1.0
    else:
        return 0.0

# 🟩 Cell 5 — Utility Functions

In [5]:
def get_distance(p1, p2):
    return np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def get_thumb_angle(lm):
    dx = lm[4].x - lm[2].x
    dy = lm[4].y - lm[2].y
    return np.degrees(np.arctan2(dy, dx))

# 🟩 Cell 6 — Servo Control Functions

In [6]:
#🔵 S4 — Horizontal Sweep (Base)
def control_s4(lm, frame_width):
    global prev_wrist_x

    x = lm[0].x * frame_width

    if prev_wrist_x is None:
        prev_wrist_x = x
        return

    delta = x - prev_wrist_x
    prev_wrist_x = x

    servo["S4"] += delta * 0.4
    servo["S4"] = np.clip(servo["S4"], 0, 180)

#🔴 S3 — Depth (Fist In/Out)
def control_s3(lm):
    global prev_depth

    depth = lm[0].z

    if prev_depth is None:
        prev_depth = depth
        return

    delta = depth - prev_depth
    prev_depth = depth

    servo["S3"] -= delta * 200
    servo["S3"] = np.clip(servo["S3"], 0, 180)

#🟡 S2 — Thumb Rotation
def control_s2(lm):
    global prev_thumb_angle

    angle = get_thumb_angle(lm)

    if prev_thumb_angle is None:
        prev_thumb_angle = angle
        return

    delta = angle - prev_thumb_angle
    prev_thumb_angle = angle

    servo["S2"] += delta
    servo["S2"] = np.clip(servo["S2"], 0, 180)

#🟢 S1 — Pinch (Gripper)
def control_s1(lm):
    global prev_pinch_dist

    dist = get_distance(lm[4], lm[8])

    if prev_pinch_dist is None:
        prev_pinch_dist = dist
        return

    delta = dist - prev_pinch_dist
    prev_pinch_dist = dist

    servo["S1"] += delta * 300
    servo["S1"] = np.clip(servo["S1"], 0, 180)

# 🟩 Cell 7 — Main Loop (FINAL STATE MACHINE)

In [7]:
def draw_landmarks(frame, landmarks):
    h, w, _ = frame.shape

    # Draw points
    for lm in landmarks:
        cx, cy = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (cx, cy), 5, (0, 255, 255), -1)

    # Draw connections (hand skeleton)
    connections = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20)
    ]

    for c in connections:
        x1, y1 = int(landmarks[c[0]].x * w), int(landmarks[c[0]].y * h)
        x2, y2 = int(landmarks[c[1]].x * w), int(landmarks[c[1]].y * h)

        cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

In [8]:
cap = cv2.VideoCapture(0)

# Optional: improve resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    # Default scores (for display safety)
    scores = {"S1": 0, "S2": 0, "S3": 0, "S4": 0}

    if result.hand_landmarks:

        hands = result.hand_landmarks
        handedness = result.handedness

        left_hand = None
        right_hand = None

        # 🧠 Assign LEFT and RIGHT hands
        for i in range(len(hands)):
            label = handedness[i][0].category_name

            if label == "Left":
                left_hand = hands[i]
            elif label == "Right":
                right_hand = hands[i]

        # 🎨 Draw BOTH hands
        if left_hand:
            draw_landmarks(frame, left_hand)

        if right_hand:
            draw_landmarks(frame, right_hand)

        # 🏷️ Label hands
        if left_hand:
            x = int(left_hand[0].x * w)
            y = int(left_hand[0].y * h)
            cv2.putText(frame, "LEFT (CONTROL)", (x, y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        if right_hand:
            x = int(right_hand[0].x * w)
            y = int(right_hand[0].y * h)
            cv2.putText(frame, "RIGHT (LOCK)", (x, y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

        # 🔒 LOCK (RIGHT HAND ONLY)
        if right_hand and is_four_fingers(right_hand) and cooldown == 0:
            lock_mode = True
            active_servo = None
            cooldown = 20

        # 🎮 CONTROL (LEFT HAND ONLY)
        if left_hand:
            lm = left_hand

            # 🎯 Compute gesture scores
            scores = {
                "S1": score_pinch(lm),
                "S3": score_three_fingers(lm),
                "S4": score_open(lm),
                "S2": score_thumb(lm)
            }

            best_gesture = max(scores, key=scores.get)
            confidence = scores[best_gesture]

            # 🔓 Unlock using confidence
            if lock_mode:
                if confidence > 0.6:
                    active_servo = best_gesture
                    lock_mode = False

            # 🎮 Normal operation
            if not lock_mode:
                if confidence > 0.6:
                    active_servo = best_gesture

                if active_servo == "S4":
                    control_s4(lm, w)

                elif active_servo == "S3":
                    control_s3(lm)

                elif active_servo == "S2":
                    control_s2(lm)

                elif active_servo == "S1":
                    control_s1(lm)

    # ⏳ Cooldown
    if cooldown > 0:
        cooldown -= 1

    # 🖥️ DISPLAY SERVO VALUES
    y0 = 30
    for i, (k, v) in enumerate(servo.items()):
        cv2.putText(frame, f"{k}: {int(v)}", (10, y0 + i*30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    # STATUS TEXT
    status = "LOCKED" if lock_mode else f"ACTIVE: {active_servo}"
    cv2.putText(frame, status, (10, 180),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    # 📊 DEBUG SCORES (VERY USEFUL)
    cv2.putText(frame,
                f"P:{scores['S1']:.2f} F:{scores['S3']:.2f} O:{scores['S4']:.2f} T:{scores['S2']:.2f}",
                (10, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    cv2.imshow("Gesture Controlled Arm", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()